# Hotel Booking Data Preprocessing Pipeline

**Reference:** Jiawei Han -- Data Mining: Concepts and Techniques, Ch. 3  
**Dataset:** `hotel_booking_data.csv` (119,390 records, 36 attributes)  
**Architecture:** Functional pipeline with composable `.pipe()` chaining  

---

| Stage | Concern | Functions |
|-------|---------|----------|
| Ingestion | Local caching, raw snapshot | `download_if_absent` |
| Audit | Discrepancy detection | `audit_nulls`, `audit_anomalies` |
| Cleaning | PII scrub, dedup, domain filter, imputation | `scrub_pii_and_leakage`, `deduplicate_records`, `filter_impossible_records`, `impute_missing_values` |
| Engineering | Feature extraction, discretization, hierarchy | `extract_domain_features`, `discretize_temporal_features` |
| Integration | Chi-Square, Pearson correlation | Statistical tests |
| Reduction | PCA eigendecomposition, 2D projection | `run_pca_analysis` |
| Export | High-res figures, CSV serialization | Asset generation |

## Sel 1: Environment Setup

In [ ]:
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.facecolor": "white",
})
sns.set_style("whitegrid")

CSV_LOCAL = "hotel_booking_data.csv"
CSV_REMOTE = "https://huggingface.co/datasets/momentarek1/hotel_booking_data/resolve/main/hotel_booking_data.csv"
CSV_OUTPUT = "hotel_booking_preprocessed.csv"

PII_COLS = ["name", "email", "phone-number", "credit_card"]
LEAKAGE_COLS = ["reservation_status", "reservation_status_date"]
SCRUB_COLS = PII_COLS + LEAKAGE_COLS

NUM_FEATURES = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "booking_changes", "adr", "total_of_special_requests",
]

PCA_FEATURES = [
    "lead_time", "total_stay_nights", "adults", "children",
    "booking_changes", "adr", "days_in_waiting_list", "total_of_special_requests",
]

MONTH_MAP = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}

SEASON_MAP = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

print(f"Output directory: {FIGURES_DIR.resolve()}")

## Sel 2: Data Ingestion and Caching

In [ ]:
def download_if_absent(local_path: str, remote_url: str) -> None:
    """Fetch CSV from remote URL if not cached locally."""
    if Path(local_path).exists():
        print(f"Cache hit: {local_path}")
        return
    import requests
    print(f"Downloading from {remote_url}")
    resp = requests.get(remote_url)
    resp.raise_for_status()
    Path(local_path).write_bytes(resp.content)
    print(f"Saved to {local_path}")


download_if_absent(CSV_LOCAL, CSV_REMOTE)
df_raw = pd.read_csv(CSV_LOCAL)
print(f"df_raw loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols")

## Sel 3: Audit Engine (Discrepancy Detection)

In [ ]:
def audit_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Return a summary of columns with missing values, sorted descending."""
    counts = df.isnull().sum()
    pct = (counts / len(df) * 100).round(4)
    summary = pd.DataFrame({"missing_count": counts, "missing_pct": pct})
    return summary[summary["missing_count"] > 0].sort_values("missing_count", ascending=False)


def audit_anomalies(df: pd.DataFrame) -> Dict[str, int]:
    """Detect domain violations and structural anomalies."""
    total_guests = df["adults"] + df["children"].fillna(0) + df["babies"].fillna(0)
    scrub_cols_present = [c for c in SCRUB_COLS if c in df.columns]
    dup_count = df.drop(columns=scrub_cols_present).duplicated().sum() if scrub_cols_present else df.duplicated().sum()
    return {
        "adr_negative": int((df["adr"] < 0).sum()),
        "adr_extreme_gt1000": int((df["adr"] > 1000).sum()),
        "zero_guests": int((total_guests == 0).sum()),
        "real_duplicates": int(dup_count),
    }


print("--- Missing Values (df_raw) ---")
display(audit_nulls(df_raw))

print("\n--- Anomaly Detection (df_raw) ---")
anomalies = audit_anomalies(df_raw)
for k, v in anomalies.items():
    print(f"  {k}: {v:,}")

## Sel 4: Modular Preprocessing Pipeline

In [ ]:
def scrub_pii_and_leakage(df: pd.DataFrame) -> pd.DataFrame:
    """Remove PII and target-leakage columns."""
    cols = [c for c in SCRUB_COLS if c in df.columns]
    out = df.drop(columns=cols)
    print(f"[scrub] Dropped {len(cols)} cols: {cols}")
    return out


def deduplicate_records(df: pd.DataFrame) -> pd.DataFrame:
    """Remove exact duplicate rows."""
    n_before = len(df)
    out = df.drop_duplicates()
    n_dropped = n_before - len(out)
    print(f"[dedup] Removed {n_dropped:,} duplicate rows")
    return out


def filter_impossible_records(df: pd.DataFrame) -> pd.DataFrame:
    """Filter rows violating domain constraints: negative/extreme ADR, zero guests."""
    total_guests = df["adults"] + df["children"].fillna(0) + df["babies"].fillna(0)
    mask = (df["adr"] >= 0) & (df["adr"] <= 1000) & (total_guests > 0)
    out = df[mask].copy()
    n_dropped = len(df) - len(out)
    print(f"[filter] Removed {n_dropped:,} impossible records")
    return out


def impute_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing values with domain-appropriate defaults."""
    out = df.copy()
    out["children"] = out["children"].fillna(0).astype(int)
    out["country"] = out["country"].fillna("Unknown")
    out["agent"] = out["agent"].fillna(0)
    out["company"] = out["company"].fillna(0)
    remaining = out.isnull().sum().sum()
    print(f"[impute] Remaining nulls: {remaining}")
    return out


def extract_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive composite features: total stay nights and room mismatch indicator."""
    out = df.copy()
    out["total_stay_nights"] = out["stays_in_weekend_nights"] + out["stays_in_week_nights"]
    out["is_room_changed"] = (out["reserved_room_type"] != out["assigned_room_type"]).astype(int)
    print(f"[features] Added total_stay_nights, is_room_changed")
    return out


def discretize_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Bin lead_time into quartiles; generate arrival date hierarchy."""
    out = df.copy()

    out["lead_time_category"] = pd.qcut(
        out["lead_time"], q=4,
        labels=["Very Short", "Short", "Medium", "Long"],
    )

    month_num = out["arrival_date_month"].map(MONTH_MAP)
    out["arrival_date"] = pd.to_datetime(
        out["arrival_date_year"].astype(str)
        + "-" + month_num.astype(str).str.zfill(2)
        + "-" + out["arrival_date_day_of_month"].astype(str).str.zfill(2)
    )
    out["arrival_quarter"] = "Q" + out["arrival_date"].dt.quarter.astype(str)
    out["arrival_season"] = month_num.map(SEASON_MAP)

    print(f"[temporal] Added lead_time_category, arrival_date, arrival_quarter, arrival_season")
    return out


df_clean = (
    df_raw
    .pipe(scrub_pii_and_leakage)
    .pipe(deduplicate_records)
    .pipe(filter_impossible_records)
    .pipe(impute_missing_values)
    .pipe(extract_domain_features)
    .pipe(discretize_temporal_features)
)

print(f"\nPipeline complete: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} cols")

## Sel 5: Data Integration and Statistical Tests

In [ ]:
# --- Chi-Square Test: reserved_room_type vs assigned_room_type ---
observed = pd.crosstab(df_clean["reserved_room_type"], df_clean["assigned_room_type"])
chi2_stat, p_value, dof, expected = chi2_contingency(observed)

print("--- Chi-Square Test ---")
print(f"  chi2     = {chi2_stat:,.4f}")
print(f"  p-value  = {p_value:.4e}")
print(f"  dof      = {dof}  [(r-1)*(c-1) = ({observed.shape[0]}-1)*({observed.shape[1]}-1)]")

print("\n--- Observed Contingency Table ---")
display(observed)

print("\n--- Expected Frequencies ---")
expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
display(expected_df.round(2))

# --- Pearson Correlation Matrix ---
corr_matrix = df_clean[NUM_FEATURES].corr()
print("\n--- Pearson Correlation Matrix ---")
display(corr_matrix.round(4))

top_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .unstack()
    .dropna()
    .sort_values(ascending=False)
    .head(5)
)
print("\n--- Top 5 Pairwise Correlations ---")
display(top_corr)

### Evaluasi Redundansi

**Nominal (Chi-Square):**
- `reserved_room_type` dan `assigned_room_type` memiliki asosiasi statistik sangat kuat (chi2 = 387,036; p < 0.001).
- Kedua atribut tetap dipertahankan karena perbedaannya merepresentasikan fenomena bisnis room upgrade/downgrade yang menjadi prediktor kuat pembatalan.

**Numerik (Pearson):**
- Korelasi tertinggi: `stays_in_weekend_nights` vs `stays_in_week_nights` (r = 0.550), tergolong korelasi sedang.
- Merujuk pada Jiawei Han (Ch. 3), redundansi baru terjadi pada |r| >= 0.85 (seperti contoh Stock A vs Stock B, r = 0.94). Tidak ada pasangan atribut numerik yang melewati ambang batas tersebut.
- Seluruh 8 atribut numerik dipertahankan.

## Sel 6: Dimensionality Reduction (PCA)

In [ ]:
def run_pca_analysis(
    df: pd.DataFrame, features: List[str]
) -> Tuple[np.ndarray, PCA, StandardScaler]:
    """Standardize features and fit full PCA. Returns projected data, fitted PCA, and scaler."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[features])
    pca = PCA()
    X_pca = pca.fit_transform(X_scaled)
    return X_pca, pca, scaler


X_pca, pca_model, scaler = run_pca_analysis(df_clean, PCA_FEATURES)

evr = pca_model.explained_variance_ratio_
cum_evr = np.cumsum(evr) * 100

pca_summary = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(PCA_FEATURES))],
    "Eigenvalue": pca_model.explained_variance_.round(4),
    "Variance_Ratio_Pct": (evr * 100).round(2),
    "Cumulative_Pct": cum_evr.round(2),
})

print("--- PCA Variance Summary ---")
display(pca_summary)

n_components_70 = int(np.argmax(cum_evr >= 70)) + 1
print(f"\nComponents needed for >=70%% variance retention: {n_components_70} (cumulative: {cum_evr[n_components_70 - 1]:.2f}%)")

### Interpretasi PCA

- PC1 menjelaskan 20.07% varians, PC2 menjelaskan 16.42%. Akumulasi 2 komponen: 36.50%.
- Threshold retention 70% tercapai pada PC5 (74.06% kumulatif).
- Proyeksi subruang 2D (PC1 vs PC2) menunjukkan overlap signifikan antara kelas `is_canceled` = 0 dan 1, mengindikasikan bahwa separasi linier 2D tidak memadai. Pemodelan prediktif memerlukan fitur kategorikal tambahan (`deposit_type`, `market_segment`, `is_room_changed`) atau algoritma non-linier (Random Forest, XGBoost).

## Sel 7: Asset Export and Final Serialization

In [ ]:
# --- Figure 1: Before vs After Cleaning ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

missing_cols = ["company", "agent", "country", "children"]
raw_miss = df_raw[missing_cols].isnull().sum()
clean_miss = df_clean[missing_cols].isnull().sum()
pd.DataFrame({"Before": raw_miss, "After": clean_miss}).plot(
    kind="bar", ax=axes[0], color=["#c0392b", "#27ae60"], alpha=0.85,
)
axes[0].set_title("(A) Missing Values per Column", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(missing_cols, rotation=0)

bp = axes[1].boxplot(
    [df_raw["adr"].dropna(), df_clean["adr"]],
    tick_labels=["Before", "After"],
    patch_artist=True,
)
bp["boxes"][0].set_facecolor("#c0392b")
bp["boxes"][1].set_facecolor("#27ae60")
for box in bp["boxes"]:
    box.set_alpha(0.7)
axes[1].set_title("(B) ADR Distribution", fontweight="bold")
axes[1].set_ylabel("Average Daily Rate")

fig.suptitle("Figure 1: Data Cleaning (Before vs. After)", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig1_before_after_cleaning.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Exported: fig1_before_after_cleaning.png")


# --- Figure 2: Data Integration ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(observed, ax=axes[0], annot=True, fmt="d", cmap="YlGnBu")
axes[0].set_title("(A) Contingency: Reserved vs Assigned Room", fontweight="bold")

sns.heatmap(corr_matrix, ax=axes[1], annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_title("(B) Pearson Correlation Matrix", fontweight="bold")

fig.suptitle("Figure 2: Data Integration & Redundancy Analysis", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig2_data_integration.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Exported: fig2_data_integration.png")


# --- Figure 3: PCA ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pc_labels = [f"PC{i+1}" for i in range(len(PCA_FEATURES))]
axes[0].plot(pc_labels, cum_evr, color="#c0392b", marker="o", linewidth=2)
axes[0].axhline(y=70, color="black", linestyle="--", linewidth=1, label="70% threshold")
axes[0].set_title("(A) Cumulative Explained Variance", fontweight="bold")
axes[0].set_ylabel("Cumulative Variance (%)")
axes[0].set_ylim(0, 105)
axes[0].legend()

scatter_df = pd.DataFrame({
    "PC1": X_pca[:, 0], "PC2": X_pca[:, 1],
    "is_canceled": df_clean["is_canceled"].values,
})
for label, color in [(0, "#27ae60"), (1, "#c0392b")]:
    subset = scatter_df[scatter_df["is_canceled"] == label]
    axes[1].scatter(
        subset["PC1"], subset["PC2"],
        c=color, alpha=0.4, s=10,
        label=f"{'Not Canceled' if label == 0 else 'Canceled'}",
    )
axes[1].set_title("(B) PCA 2D Projection (PC1 vs PC2)", fontweight="bold")
axes[1].set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
axes[1].legend(markerscale=3)

fig.suptitle("Figure 3: Dimensionality Reduction (PCA)", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig3_data_reduction_pca.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Exported: fig3_data_reduction_pca.png")


# --- Figure 4: Discretization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

eq_width = pd.cut(df_clean["lead_time"], bins=4)
w_counts = eq_width.value_counts().sort_index()

lt_labels = ["Very Short", "Short", "Medium", "Long"]
f_counts = df_clean["lead_time_category"].value_counts()[lt_labels]

axes[0].bar(range(4), w_counts.values, color="#c0392b", alpha=0.85)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([f"Bin {i+1}" for i in range(4)])
axes[0].set_title("(A) Equal-Width Partitioning", fontweight="bold")
axes[0].set_ylabel("Frequency")
for i, v in enumerate(w_counts.values):
    axes[0].text(i, v + 800, f"{v:,}", ha="center", fontweight="bold", fontsize=9)

axes[1].bar(lt_labels, f_counts.values, color="#27ae60", alpha=0.85)
axes[1].set_title("(B) Equal-Frequency Partitioning", fontweight="bold")
axes[1].set_ylabel("Frequency")
for i, v in enumerate(f_counts.values):
    axes[1].text(i, v + 400, f"{v:,}", ha="center", fontweight="bold", fontsize=9)

fig.suptitle("Figure 4: Discretization -- Lead Time Binning Comparison", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_discretization_binning.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Exported: fig4_discretization_binning.png")


# --- Quantitative Comparison Table ---
raw_anom = audit_anomalies(df_raw)

comparison = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Attributes",
        "Total Missing Cells",
        "Real Duplicate Rows",
        "Impossible/Extreme ADR",
        "Ghost Bookings (0 Guests)",
    ],
    "Before (df_raw)": [
        f"{len(df_raw):,}",
        str(df_raw.shape[1]),
        f"{df_raw.isnull().sum().sum():,}",
        f"{raw_anom['real_duplicates']:,}",
        f"{raw_anom['adr_negative'] + raw_anom['adr_extreme_gt1000']:,}",
        f"{raw_anom['zero_guests']:,}",
    ],
    "After (df_clean)": [
        f"{len(df_clean):,}",
        str(df_clean.shape[1]),
        "0",
        "0",
        "0",
        "0",
    ],
})

print("\n--- Before vs After (Markdown) ---")
print(comparison.to_markdown(index=False))
display(comparison)


# --- Serialize final dataset ---
df_clean.to_csv(CSV_OUTPUT, index=False)
print(f"\nSerialized: {CSV_OUTPUT} ({df_clean.shape[0]:,} rows x {df_clean.shape[1]} cols)")